# ShaadiSahulat — Visual Dress Recommendation: Model Training
### FYP 2026 | NUCES Chiniot-Faisalabad

---

**What this notebook does:**
- Fine-tunes **EfficientNet-B0** (5M params, 3× faster than ResNet50) on your dress images
- Two training phases: Classification → Triplet Loss refinement
- Outputs one file you copy to your local PC: `fine_tuned_efficientnet_b0.pth`

**Your 3 categories:**

| Folder Name | Label |
|---|---|
| `bridal_lehenga` | Bridal Lehenga |
| `bridal_sharara` | Bridal Sharara |
| `bridal_saree` | Bridal Saree |

**Minimum images per category:** 10 (recommended: 20+)

---

### BEFORE YOU START:
1. Go to **Runtime → Change runtime type → T4 GPU** (free, ~1 hr training)
2. Run cells **top to bottom** — do not skip any cell
3. At the end, download `fine_tuned_efficientnet_b0.pth` and put it in `visual-ml-service/models/`

---
## Cell 1 — Install Packages & Check GPU

In [ ]:
!pip install -q torch torchvision scikit-learn Pillow matplotlib numpy

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime -> Change runtime type -> T4 GPU')

---
## Cell 2 — Upload Your Training Images

**Choose ONE option below.** Comment out the one you don't use.

### Option A — Upload a ZIP file (easiest)
Create a ZIP file on your PC with this structure:
```
training_data.zip
├── bridal_lehenga/
│   ├── bridal_lehenga_001.jpg
│   ├── bridal_lehenga_002.jpg
│   └── ... (10–20 images)
├── bridal_sharara/
│   └── ... (10–20 images)
└── bridal_saree/
    └── ... (10–20 images)
```
How to make the ZIP on Windows:  
Select the 3 folders → Right-click → Send to → Compressed (zipped) folder  
Name it `training_data.zip`

### Option B — Use Google Drive
Upload the `training_data/` folder to `My Drive/shaadi-sahulat/training_data/`  
Then use the Drive mount code.

In [ ]:
import os, zipfile

# ============================================================
# OPTION A: Upload ZIP file
# ============================================================
# Run this cell, click 'Choose Files', and select training_data.zip

from google.colab import files as colab_files
uploaded = colab_files.upload()          # a file picker will appear

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

TRAIN_DIR = '/content/training_data'
print(f'Extracted to {TRAIN_DIR}')
print('Contents:', os.listdir(TRAIN_DIR))

In [ ]:
# ============================================================
# OPTION B: Google Drive (skip if you used Option A)
# ============================================================
# Uncomment and run if your images are on Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# TRAIN_DIR = '/content/drive/MyDrive/shaadi-sahulat/training_data'
# print('Drive mounted. Training dir:', TRAIN_DIR)

---
## Cell 3 — Verify Dataset

In [ ]:
from PIL import Image

CATEGORIES = ['bridal_lehenga', 'bridal_sharara', 'bridal_saree']
VALID_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

print('=' * 60)
print('  DATASET VERIFICATION')
print('=' * 60)

total_images = 0
all_valid    = True

for cat in CATEGORIES:
    cat_dir = os.path.join(TRAIN_DIR, cat)
    if not os.path.exists(cat_dir):
        print(f'  MISSING  {cat}  ← folder not found!')
        all_valid = False
        continue

    files = [f for f in os.listdir(cat_dir)
             if os.path.splitext(f)[1].lower() in VALID_EXTS]
    count  = len(files)
    total_images += count

    icon = '✅' if count >= 20 else ('⚠️ ' if count >= 10 else '❌')
    if count < 10:
        all_valid = False

    print(f'  {icon}  {cat:22s} : {count:3d} images', end='')
    if count >= 10:
        try:
            sample = Image.open(os.path.join(cat_dir, files[0]))
            print(f'  (sample size: {sample.size[0]}x{sample.size[1]})', end='')
        except:
            pass
    print()

print()
print(f'  Total images       : {total_images}')
print(f'  After augmentation : ~{total_images * 10} samples')
print(f'  Ready for training : {"YES" if all_valid else "NO — fix issues above"}')
print('=' * 60)

if not all_valid:
    raise SystemExit('Fix dataset issues before continuing.')

---
## Cell 4 — Data Loader with Augmentation

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF

# ── Constants ──────────────────────────────────────────────
IMAGE_SIZE             = 224
AUGMENTATION_MULTIPLIER = 10
VALIDATION_SPLIT       = 0.2
BATCH_SIZE             = 16
NUM_CLASSES            = 3
EMBEDDING_DIM          = 128

class DressDataset(Dataset):
    def __init__(self, image_paths, labels, augment=False):
        self.image_paths = image_paths
        self.labels      = labels
        self.augment     = augment
        self.base_transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            ),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.augment:
            image = image.resize((int(IMAGE_SIZE * 1.2), int(IMAGE_SIZE * 1.2)))
            if np.random.random() < 0.5:
                image = TF.hflip(image)
            image = TF.rotate(image, float(np.random.uniform(-15, 15)))
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                image, scale=(0.8, 1.0), ratio=(0.75, 1.33))
            image = TF.crop(image, i, j, h, w)
            image = transforms.ColorJitter(
                brightness=0.2, contrast=0.2, saturation=0.2)(image)
            image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
        return self.base_transform(image), self.labels[idx]


# Build train / val splits
train_paths, train_labels = [], []
val_paths,   val_labels   = [], []

for label_idx, cat in enumerate(CATEGORIES):
    cat_dir = os.path.join(TRAIN_DIR, cat)
    all_files = sorted([
        os.path.join(cat_dir, f) for f in os.listdir(cat_dir)
        if os.path.splitext(f)[1].lower() in VALID_EXTS
    ])
    n_val     = max(1, int(len(all_files) * VALIDATION_SPLIT))
    val_files = all_files[:n_val]
    trn_files = all_files[n_val:]

    for fp in val_files:
        val_paths.append(fp);  val_labels.append(label_idx)
    for fp in trn_files:
        for _ in range(AUGMENTATION_MULTIPLIER):
            train_paths.append(fp);  train_labels.append(label_idx)

train_loader = DataLoader(
    DressDataset(train_paths, train_labels, augment=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    DressDataset(val_paths, val_labels, augment=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f'Train : {len(train_paths):4d} samples  '
      f'({len(train_paths)//AUGMENTATION_MULTIPLIER} originals × {AUGMENTATION_MULTIPLIER} augmented)')
print(f'Val   : {len(val_paths):4d} samples  (no augmentation)')
print(f'Batch size : {BATCH_SIZE}')
print(f'Batches/epoch (train): {len(train_loader)}')

---
## Cell 5 — Model: EfficientNet-B0 (128-dim Embedding)

In [ ]:
import torch.nn as nn
import torchvision.models as models

class DressEmbeddingModel(nn.Module):
    """
    EfficientNet-B0 fine-tuned for dress embedding.
    Architecture matches visual-ml-service/model.py exactly.
    """
    def __init__(self, num_classes=NUM_CLASSES, embedding_dim=EMBEDDING_DIM):
        super().__init__()

        # Load pretrained EfficientNet-B0
        base = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )

        # Freeze all layers
        for param in base.parameters():
            param.requires_grad = False

        # Unfreeze last 3 feature blocks (features.6, .7, .8)
        for name, param in base.named_parameters():
            if any(blk in name for blk in ['features.6', 'features.7', 'features.8']):
                param.requires_grad = True

        in_features = base.classifier[1].in_features   # 1280
        base.classifier = nn.Identity()
        self.backbone = base

        # Embedding head: 1280 → 512 → 128
        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, embedding_dim),
        )

        # Classifier head (used only during training)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        """Returns (logits, embedding) — used during training."""
        features  = self.backbone(x)
        embedding = self.embedding(features)
        logits    = self.classifier(embedding)
        return logits, embedding

    def get_embedding(self, x):
        """Returns L2-normalised 128-dim embedding — used at inference."""
        features  = self.backbone(x)
        embedding = self.embedding(features)
        return nn.functional.normalize(embedding, p=2, dim=1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DressEmbeddingModel().to(device)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Backbone          : EfficientNet-B0')
print(f'Total parameters  : {total_p:,}')
print(f'Trainable params  : {trainable_p:,}  (unfrozen blocks: features.6/7/8 + head)')
print(f'Frozen params     : {total_p - trainable_p:,}')
print(f'Embedding dim     : {EMBEDDING_DIM}')
print(f'Num classes       : {NUM_CLASSES}')
print(f'Device            : {device}')

---
## Cell 6 — Phase 1: Classification Pre-Training

**Expected time on T4 GPU:** ~15–30 minutes  
**Target:** Validation accuracy > 80%  
Early stopping stops automatically when val loss stops improving.

In [ ]:
import time, json
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

MAX_EPOCHS = 30
PATIENCE   = 5

history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_loss    = float('inf')
patience_counter = 0

print('PHASE 1 — Classification Pre-Training')
print('=' * 72)

for epoch in range(MAX_EPOCHS):
    t0 = time.time()

    # ── Train ──
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(imgs)
        loss      = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        run_loss += loss.item()
        preds     = logits.argmax(dim=1)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)

    train_loss = run_loss / len(train_loader)
    train_acc  = 100.0 * correct / total

    # ── Validate ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits, _    = model(imgs)
            v_loss      += criterion(logits, labels).item()
            preds        = logits.argmax(dim=1)
            v_correct   += (preds == labels).sum().item()
            v_total     += labels.size(0)

    val_loss = v_loss / len(val_loader)
    val_acc  = 100.0 * v_correct / v_total
    scheduler.step(val_loss)

    history['train_loss'].append(round(train_loss, 4))
    history['train_accuracy'].append(round(train_acc, 2))
    history['val_loss'].append(round(val_loss, 4))
    history['val_accuracy'].append(round(val_acc, 2))

    elapsed = time.time() - t0
    ckpt_mark = ''
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'fine_tuned_efficientnet_b0.pth')
        ckpt_mark = '  ← saved'
    else:
        patience_counter += 1

    print(f'  Ep {epoch+1:2d}/{MAX_EPOCHS} │ '
          f'Train loss={train_loss:.4f} acc={train_acc:.1f}% │ '
          f'Val loss={val_loss:.4f} acc={val_acc:.1f}% │ '
          f'{elapsed:.0f}s{ckpt_mark}')

    if patience_counter >= PATIENCE:
        print(f'\n  Early stop — val loss flat for {PATIENCE} epochs.')
        break

best_acc = max(history['val_accuracy'])
print(f'\nPhase 1 complete.  Best val accuracy: {best_acc:.1f}%')
if best_acc < 70:
    print('Tip: accuracy < 70% — try adding more images (20+ per category).')

---
## Cell 7 — Phase 2: Triplet Loss Embedding Refinement

Pulls same-category dresses **closer** in embedding space.  
Pushes different-category dresses **farther** apart.  
Makes the 128-dim similarity search more accurate.  

**Expected time:** ~3–5 minutes on T4 GPU

In [ ]:
# Load best Phase 1 weights before refinement
model.load_state_dict(torch.load('fine_tuned_efficientnet_b0.pth', map_location=device))

class TripletLoss(nn.Module):
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin
    def forward(self, anchor, positive, negative):
        d_pos = torch.norm(anchor - positive, p=2, dim=1)
        d_neg = torch.norm(anchor - negative, p=2, dim=1)
        return torch.clamp(d_pos - d_neg + self.margin, min=0.0).mean()

triplet_loss_fn  = TripletLoss(margin=0.2)
triplet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5
)

TRIPLET_EPOCHS = 10
N_TRIPLETS     = 100

print('PHASE 2 — Triplet Loss Embedding Refinement')
print('=' * 72)

# Pre-compute embeddings on training set
model.eval()
all_embs, all_lbls = [], []
with torch.no_grad():
    for imgs, labels in train_loader:
        all_embs.append(model.get_embedding(imgs.to(device)).cpu())
        all_lbls.extend(labels.numpy())

all_embs = torch.cat(all_embs, dim=0)
all_lbls = np.array(all_lbls)
n        = len(all_lbls)

model.train()
for epoch in range(TRIPLET_EPOCHS):
    epoch_loss = 0.0
    for _ in range(N_TRIPLETS):
        a_idx  = np.random.randint(0, n)
        a_lbl  = all_lbls[a_idx]
        pos_ids = np.where(all_lbls == a_lbl)[0]
        neg_ids = np.where(all_lbls != a_lbl)[0]
        if len(pos_ids) < 2 or len(neg_ids) < 1:
            continue
        p_idx = np.random.choice(pos_ids)
        n_idx = np.random.choice(neg_ids)

        anchor   = all_embs[a_idx].unsqueeze(0).to(device)
        positive = all_embs[p_idx].unsqueeze(0).to(device)
        negative = all_embs[n_idx].unsqueeze(0).to(device)

        triplet_optimizer.zero_grad()
        loss = triplet_loss_fn(anchor, positive, negative)
        loss.backward()
        triplet_optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / N_TRIPLETS
    print(f'  Triplet epoch {epoch+1:2d}/{TRIPLET_EPOCHS} │ loss={avg:.4f}')

# Save final model (Phase 1 + Phase 2)
torch.save(model.state_dict(), 'fine_tuned_efficientnet_b0.pth')
print('\nPhase 2 complete. final model saved to fine_tuned_efficientnet_b0.pth')

---
## Cell 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'],     label='Train Loss',     color='#d946ef', linewidth=2)
axes[0].plot(history['val_loss'],       label='Val Loss',       color='#f59e0b', linewidth=2)
axes[0].set_title('Loss — EfficientNet-B0 Fine-Tuning', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_accuracy'], label='Train Accuracy', color='#d946ef', linewidth=2)
axes[1].plot(history['val_accuracy'],   label='Val Accuracy',   color='#10b981', linewidth=2)
axes[1].set_title('Accuracy — EfficientNet-B0 Fine-Tuning', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

---
## Cell 9 — Save All Files + Download

This cell saves and downloads everything you need.  
**After downloading, copy these files to your PC:**

| Downloaded File | Copy to (on your PC) |
|---|---|
| `fine_tuned_efficientnet_b0.pth` | `visual-ml-service/models/` |
| `category_classifier.pth` | `visual-ml-service/models/` |
| `training_history.json` | `visual-ml-service/models/` |
| `class_names.json` | `visual-ml-service/models/` |
| `training_curves.png` | anywhere (optional, just for reference) |

In [ ]:
from google.colab import files as colab_files

# Save helper files
torch.save(model.classifier.state_dict(), 'category_classifier.pth')

with open('training_history.json', 'w') as fh:
    json.dump(history, fh, indent=2)

with open('class_names.json', 'w') as fh:
    json.dump(CATEGORIES, fh, indent=2)

# Report
print('Files saved:')
for fname in ['fine_tuned_efficientnet_b0.pth', 'category_classifier.pth',
              'training_history.json', 'class_names.json']:
    size = os.path.getsize(fname) / 1024
    print(f'  {fname:40s}  {size:7.0f} KB')

best_val = max(history['val_accuracy'])
print(f'\nBest validation accuracy : {best_val:.1f}%')
print(f'Model backbone           : EfficientNet-B0')
print(f'Categories               : {CATEGORIES}')

print('\nDownloading files...')
for fname in ['fine_tuned_efficientnet_b0.pth', 'category_classifier.pth',
              'training_history.json', 'class_names.json', 'training_curves.png']:
    if os.path.exists(fname):
        colab_files.download(fname)
        print(f'  Downloaded: {fname}')

print('\nDone! Now copy the .pth and .json files to visual-ml-service/models/ on your PC.')

---
## What to Do Next (on your local PC)

### Step 1: Copy files to `visual-ml-service/models/`
```
visual-ml-service/
└── models/
    ├── fine_tuned_efficientnet_b0.pth   ← main model (required)
    ├── category_classifier.pth          ← classifier head
    ├── training_history.json
    └── class_names.json
```

### Step 2: Fill in descriptions.json
Open `visual-ml-service/descriptions.json` and write a description for each image:
```json
"bridal_lehenga_001.jpg": "deep red bridal lehenga with heavy gold embroidery"
```

### Step 3: Seed the catalog (one time)
```powershell
cd visual-ml-service
myvenv\Scripts\activate
python seed_catalog.py
```

### Step 4: Start the service
```powershell
python app.py
```

### Step 5: Test
```powershell
curl http://localhost:5002/health
```
You should see `"model_trained": true`

---
**FYP 2026 | NUCES Chiniot-Faisalabad | ShaadiSahulat**